In [1]:

# Data handling
import pandas as pd
import numpy as np

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Fairness evaluation
from fairlearn.metrics import (
    demographic_parity_difference,
    equalized_odds_difference,
    true_positive_rate,
    false_positive_rate
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# General utilities
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("Stage 5 libraries imported successfully.")


Stage 5 libraries imported successfully.


In [2]:

# Stage 3: Data imbalance strategies
imbalance_strategies = [
    "Original",
    "Class Weight",
    "SMOTENC"
]

# Seven ML models
model_names = [
    "Logistic Regression",
    "KNN",
    "Naive Bayes",
    "SVM",
    "Decision Tree",
    "Random Forest",
    "XGBoost"
]

# Stage 4: Protected attributes and group dimensions
sensitive_attributes = [
    "SEX",
    "AGE",
    "EDUCATION",
    "MARRIAGE"
]

fairness_groups = [
    "Gender",
    "Age",
    "Education",
    "Marital Status",
    "Gender × Age"
]

# Seven fairness dimensions
fairness_dimensions = [
    "Demographic Parity",
    "Equal Opportunity",
    "Calibration Bias",
    "Disparate Impact",
    "Counterfactual Fairness",
    "Intersectional Bias",
    "Individual Fairness"
]

print("Stage 5 configuration completed.")
print("Models:", len(model_names))
print("Imbalance strategies:", len(imbalance_strategies))
print("Fairness groups:", len(fairness_groups))
print("Fairness dimensions:", len(fairness_dimensions))


Stage 5 configuration completed.
Models: 7
Imbalance strategies: 3
Fairness groups: 5
Fairness dimensions: 7


In [3]:

# Store experiment results
baseline_results = []
fairness_results = []
mitigation_results = []

# Store predictions for later fairness analysis
model_predictions = {}

# Store trained models
trained_models = {}

# Protected group definitions
group_columns = {
    "Gender": "SEX",
    "Age": "AGE",
    "Education": "EDUCATION",
    "Marital Status": "MARRIAGE"
}

print("Experiment structure initialized.")
print("Results and predictions storage is ready.")


Experiment structure initialized.
Results and predictions storage is ready.


In [4]:

import os

# Check the current notebook directory
print("Current directory:", os.getcwd())

# List available files and folders
for item in os.listdir():
    print(item)


Current directory: C:\Users\Aryan
.anaconda
.antigravity-ide
.arduinoIDE
.cache
.conda
.config
.continuum
.copilot
.docker
.equo
.gemini
.ghcp-appmod
.ghcp-appmod-java
.gitconfig
.gradle
.ipynb_checkpoints
.ipython
.jupyter
.kaggle
.keras
.knime
.lakectl.yaml
.lesshst
.matplotlib
.mcp-auth
.node_repl_history
.python_history
.securecoder
.viminfo
.virtual_documents
.vscode
.vscode-shared
01_data_preprocessing.ipynb
02_model_training.ipynb
03_imbalance_experiment.ipynb
04_bias_detection_fairness_audit.ipynb
05_fairness_mitigation.ipynb
anaconda3
anaconda_projects
AppData
Application Data
Contacts
Cookies
creditcard.csv
Decision_Tree_Classification.ipynb
Decision_Tree_Regression.ipynb
Desktop
Documents
Downloads
Favorites
Hierarchical_Agglomerative.ipynb
iris-model-training.ipynb
KMeans.ipynb
knn.ipynb
lakefs
lakefs_backup
lakefs_old
LinearRegression.ipynb
Links
Local Settings
logistic.ipynb
Markov.ipynb
mlartifacts
mlflow.db
mlruns
ML_Lab_1.ipynb
Music
My Documents
Naive_Bayes.ipynb
NetH

In [5]:

import os

# Search for Stage 3-related files in the current directory
for root, dirs, files in os.walk(os.getcwd()):
    # Avoid searching unnecessary hidden/system folders
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".") and d not in [
            "anaconda3", "AppData", "OneDrive"
        ]
    ]

    for file in files:
        if "stage3" in file.lower() or "imbalance" in file.lower():
            print(os.path.join(root, file))


C:\Users\Aryan\03_imbalance_experiment.ipynb


In [8]:

from pathlib import Path

# Search likely project location
search_root = Path(r"D:\TY_Sem1\MLOps\FairXAi")

target_files = [
    "stage3_model_results.csv",
    "stage3_strategy_changes.csv",
    "train_smote.csv",
    "smote_metadata.json"
]

for filename in target_files:
    matches = list(search_root.rglob(filename))
    
    if matches:
        print(f"\n{filename}")
        for path in matches:
            print("Found:", path)
    else:
        print(f"\n{filename}: Not found")



stage3_model_results.csv
Found: D:\TY_Sem1\MLOps\FairXAi\data\results\stage3_model_results.csv

stage3_strategy_changes.csv
Found: D:\TY_Sem1\MLOps\FairXAi\data\results\stage3_strategy_changes.csv

train_smote.csv
Found: D:\TY_Sem1\MLOps\FairXAi\data\smote\train_smote.csv

smote_metadata.json
Found: D:\TY_Sem1\MLOps\FairXAi\data\smote\smote_metadata.json


In [9]:

from pathlib import Path
import pandas as pd
import json

# Correct FairXAI project directory
project_path = Path(r"D:\TY_Sem1\MLOps\FairXAi")

# Artifact directories
results_path = project_path / "data" / "results"
smote_path = project_path / "data" / "smote"

# Load Stage 3 artifacts
stage3_results = pd.read_csv(
    results_path / "stage3_model_results.csv"
)

strategy_changes = pd.read_csv(
    results_path / "stage3_strategy_changes.csv"
)

smote_train = pd.read_csv(
    smote_path / "train_smote.csv"
)

with open(smote_path / "smote_metadata.json", "r") as f:
    smote_metadata = json.load(f)

# Display confirmation
print("Stage 3 artifacts loaded successfully!")

print("\nStage 3 Results:", stage3_results.shape)
print("Strategy Changes:", strategy_changes.shape)
print("SMOTENC Training Data:", smote_train.shape)

print("\nStage 3 Result Columns:")
print(stage3_results.columns.tolist())

print("\nSMOTENC Class Distribution:")
print(smote_train.iloc[:, -1].value_counts())


Stage 3 artifacts loaded successfully!

Stage 3 Results: (19, 17)
Strategy Changes: (12, 8)
SMOTENC Training Data: (37382, 24)

Stage 3 Result Columns:
['Model', 'Strategy', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC', 'Log Loss', 'Brier', 'Balanced Accuracy', 'FPR', 'FNR', 'TN', 'FP', 'FN', 'TP']

SMOTENC Class Distribution:
default payment next month
0    18691
1    18691
Name: count, dtype: int64


In [10]:

# Search for existing training, validation and preprocessing files

search_folders = [
    project_path / "data",
    project_path / "artifacts"
]

keywords = [
    "train",
    "test",
    "validation",
    "val",
    "preprocessor"
]

for folder in search_folders:
    if folder.exists():
        print(f"\n--- {folder} ---")
        
        for file in folder.rglob("*"):
            if file.is_file() and any(
                word in file.name.lower() for word in keywords
            ):
                print(file)



--- D:\TY_Sem1\MLOps\FairXAi\data ---
D:\TY_Sem1\MLOps\FairXAi\data\processed\test_processed.csv
D:\TY_Sem1\MLOps\FairXAi\data\processed\train_processed.csv
D:\TY_Sem1\MLOps\FairXAi\data\results\stage5_baseline_validation.csv
D:\TY_Sem1\MLOps\FairXAi\data\results\stage5_sensitive_removed_validation.csv
D:\TY_Sem1\MLOps\FairXAi\data\smote\train_smote.csv
D:\TY_Sem1\MLOps\FairXAi\data\split\test.csv
D:\TY_Sem1\MLOps\FairXAi\data\split\train.csv

--- D:\TY_Sem1\MLOps\FairXAi\artifacts ---
D:\TY_Sem1\MLOps\FairXAi\artifacts\preprocessor.pkl


In [11]:

# Load original Stage 1 data
train_data = pd.read_csv(
    project_path / "data" / "split" / "train.csv"
)

test_data = pd.read_csv(
    project_path / "data" / "split" / "test.csv"
)

# Identify target column
target_column = "default payment next month"

# Separate features and target
X_train_full = train_data.drop(columns=[target_column])
y_train_full = train_data[target_column]

X_test = test_data.drop(columns=[target_column])
y_test = test_data[target_column]

# Check loaded data
print("Original data loaded successfully!")

print("\nTraining data:", X_train_full.shape)
print("Test data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train_full.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

print("\nTraining columns:")
print(X_train_full.columns.tolist())


Original data loaded successfully!

Training data: (24000, 23)
Test data: (6000, 23)

Training target distribution:
default payment next month
0    18691
1     5309
Name: count, dtype: int64

Test target distribution:
default payment next month
0    4673
1    1327
Name: count, dtype: int64

Training columns:
['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


In [12]:

# Create training and validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

# Reset indexes for consistency
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("Training-validation split completed!")

print("\nTraining:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nDefault distribution:")
print("Training:\n", y_train.value_counts())
print("Validation:\n", y_val.value_counts())


Training-validation split completed!

Training: (19200, 23)
Validation: (4800, 23)
Test: (6000, 23)

Default distribution:
Training:
 default payment next month
0    14953
1     4247
Name: count, dtype: int64
Validation:
 default payment next month
0    3738
1    1062
Name: count, dtype: int64


In [13]:

# Separate SMOTENC features and target
X_train_smote = smote_train.drop(columns=[target_column])
y_train_smote = smote_train[target_column]

# Ensure column order matches original training data
X_train_smote = X_train_smote[X_train.columns]

# Verify all strategies
print("Imbalance strategies prepared.")

print("\nOriginal training:", X_train.shape)
print("Class Weight training:", X_train.shape)
print("SMOTENC training:", X_train_smote.shape)

print("\nSMOTENC target distribution:")
print(y_train_smote.value_counts())

print("\nFeature columns match:",
      X_train.columns.equals(X_train_smote.columns))


Imbalance strategies prepared.

Original training: (19200, 23)
Class Weight training: (19200, 23)
SMOTENC training: (37382, 23)

SMOTENC target distribution:
default payment next month
0    18691
1    18691
Name: count, dtype: int64

Feature columns match: True


In [14]:

from imblearn.over_sampling import SMOTENC

# Categorical columns
categorical_columns = ["SEX", "EDUCATION", "MARRIAGE"]

# Find categorical feature indices
categorical_indices = [
    X_train.columns.get_loc(col)
    for col in categorical_columns
]

# Apply SMOTENC only on training data
smotenc = SMOTENC(
    categorical_features=categorical_indices,
    random_state=RANDOM_STATE
)

X_train_smote, y_train_smote = smotenc.fit_resample(
    X_train,
    y_train
)

# Reset indexes
X_train_smote = pd.DataFrame(
    X_train_smote,
    columns=X_train.columns
)

y_train_smote = pd.Series(
    y_train_smote,
    name=target_column
)

print("SMOTENC applied correctly!")

print("\nOriginal training:", X_train.shape)
print("SMOTENC training:", X_train_smote.shape)

print("\nSMOTENC class distribution:")
print(y_train_smote.value_counts())

print("\nValidation remains unchanged:", X_val.shape)


SMOTENC applied correctly!

Original training: (19200, 23)
SMOTENC training: (29906, 23)

SMOTENC class distribution:
default payment next month
0    14953
1    14953
Name: count, dtype: int64

Validation remains unchanged: (4800, 23)


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_columns = ["SEX", "EDUCATION", "MARRIAGE"]

ordinal_columns = [
    "PAY_0", "PAY_2", "PAY_3",
    "PAY_4", "PAY_5", "PAY_6"
]

numerical_columns = [
    "LIMIT_BAL", "AGE",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"),
         categorical_columns),
        ("numerical", StandardScaler(), numerical_columns),
        ("ordinal", "passthrough", ordinal_columns)
    ]
)

print("Preprocessing configuration created.")
print("Categorical features:", len(categorical_columns))
print("Numerical features:", len(numerical_columns))
print("Ordinal features:", len(ordinal_columns))

Preprocessing configuration created.
Categorical features: 3
Numerical features: 14
Ordinal features: 6


In [16]:
# Fit preprocessing on original training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Transform validation data using the same fitted preprocessor
X_val_processed = preprocessor.transform(X_val)

# Transform SMOTENC training data using the same preprocessor
X_train_smote_processed = preprocessor.transform(X_train_smote)

print("Preprocessing completed!")

print("\nOriginal training:", X_train_processed.shape)
print("Class Weight training:", X_train_processed.shape)
print("SMOTENC training:", X_train_smote_processed.shape)
print("Validation:", X_val_processed.shape)

Preprocessing completed!

Original training: (19200, 33)
Class Weight training: (19200, 33)
SMOTENC training: (29906, 33)
Validation: (4800, 33)


In [17]:
strategy_data = {
    "Original": (X_train_processed, y_train),
    "Class Weight": (X_train_processed, y_train),
    "SMOTENC": (X_train_smote_processed, y_train_smote)
}

for strategy, (X_data, y_data) in strategy_data.items():
    print(f"\n{strategy}")
    print("Features:", X_data.shape)
    print("Target distribution:")
    print(y_data.value_counts().to_dict())


Original
Features: (19200, 33)
Target distribution:
{0: 14953, 1: 4247}

Class Weight
Features: (19200, 33)
Target distribution:
{0: 14953, 1: 4247}

SMOTENC
Features: (29906, 33)
Target distribution:
{0: 14953, 1: 14953}


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("All seven model libraries imported successfully.")

All seven model libraries imported successfully.


In [19]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=21
    ),

    "Naive Bayes": GaussianNB(),

    "SVM": SVC(
        probability=True, random_state=RANDOM_STATE
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

class_weight_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "SVM": SVC(
        probability=True, class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        scale_pos_weight=14953 / 4247,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

print("Standard models:", len(models))
print("Class-weight models:", len(class_weight_models))
print("Models without class-weight support: KNN, Naive Bayes")

Standard models: 7
Class-weight models: 5
Models without class-weight support: KNN, Naive Bayes


In [20]:
from sklearn.base import clone

baseline_results = []
model_predictions = {}
trained_models = {}

for strategy, (X_data, y_data) in strategy_data.items():

    print(f"\nTraining strategy: {strategy}")

    for model_name in model_names:

        if strategy == "Class Weight":
            if model_name not in class_weight_models:
                print(f"Skipping {model_name}: Class Weight not supported")
                continue
            model = clone(class_weight_models[model_name])

        else:
            model = clone(models[model_name])

        model.fit(X_data, y_data)

        y_pred = model.predict(X_val_processed)
        y_prob = model.predict_proba(X_val_processed)[:, 1]

        tn, fp, fn, tp = confusion_matrix(
            y_val, y_pred, labels=[0, 1]
        ).ravel()

        baseline_results.append({
            "Strategy": strategy,
            "Model": model_name,
            "Accuracy": accuracy_score(y_val, y_pred),
            "Precision": precision_score(y_val, y_pred, zero_division=0),
            "Recall": recall_score(y_val, y_pred, zero_division=0),
            "F1": f1_score(y_val, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_val, y_prob),
            "FPR": fp / (fp + tn),
            "FNR": fn / (fn + tp)
        })

        model_predictions[(strategy, model_name)] = {
            "y_pred": y_pred,
            "y_prob": y_prob
        }

        trained_models[(strategy, model_name)] = model

print("\nTraining and validation evaluation completed!")

baseline_df = pd.DataFrame(baseline_results)
display(baseline_df.round(4))


Training strategy: Original

Training strategy: Class Weight
Skipping KNN: Class Weight not supported
Skipping Naive Bayes: Class Weight not supported

Training strategy: SMOTENC

Training and validation evaluation completed!


,Strategy,Model,Accuracy,Precision,Recall,F1,ROC-AUC,FPR,FNR
0,Original,Logistic Regression,0.8123,0.7048,0.2608,0.3808,0.7244,0.0310,0.7392
1,Original,KNN,0.8125,0.6441,0.3409,0.4458,0.7459,0.0535,0.6591
2,Original,Naive Bayes,0.2877,0.2325,0.9642,0.3746,0.7342,0.9045,0.0358
3,Original,SVM,0.8185,0.6812,0.3380,0.4519,0.7145,0.0449,0.6620
4,Original,Decision Tree,0.7246,0.3822,0.3974,0.3897,0.6073,0.1825,0.6026
5,Original,Random Forest,0.8181,0.6594,0.3682,0.4725,0.7720,0.0540,0.6318
6,Original,XGBoost,0.8112,0.6309,0.3540,0.4536,0.7584,0.0589,0.6460
7,Class Weight,Logistic Regression,0.6919,0.3841,0.6507,0.4830,0.7257,0.2964,0.3493
8,Class Weight,SVM,0.7738,0.4904,0.5782,0.5307,0.7599,0.1707,0.4218
9,Class Weight,Decision Tree,0.7258,0.3811,0.3832,0.3822,0.6033,0.1768,0.6168


In [21]:
baseline_df = pd.DataFrame(baseline_results)

baseline_file = results_path / "stage5_baseline_validation.csv"
baseline_df.to_csv(baseline_file, index=False)

print("Baseline results saved!")
print("Location:", baseline_file)
print("Total configurations:", len(baseline_df))

Baseline results saved!
Location: D:\TY_Sem1\MLOps\FairXAi\data\results\stage5_baseline_validation.csv
Total configurations: 19


In [22]:
def create_fairness_groups(data):

    groups = pd.DataFrame(index=data.index)

    # Gender
    groups["Gender"] = data["SEX"].map({
        1: "Male",
        2: "Female"
    })

    # Age groups
    groups["Age"] = pd.cut(
        data["AGE"],
        bins=[20, 29, 39, 49, 59, np.inf],
        labels=["21-29", "30-39", "40-49", "50-59", "60+"]
    )

    # Education
    groups["Education"] = data["EDUCATION"].map({
        1: "Graduate School",
        2: "University",
        3: "High School",
        0: "Other",
        4: "Other",
        5: "Other",
        6: "Other"
    })

    # Marital Status
    groups["Marital Status"] = data["MARRIAGE"].map({
        1: "Married",
        2: "Single",
        0: "Other",
        3: "Other"
    })

    # Intersectional group
    groups["Gender × Age"] = (
        groups["Gender"].astype(str) + " × " +
        groups["Age"].astype(str)
    )

    return groups


validation_groups = create_fairness_groups(X_val)

for column in validation_groups.columns:
    print(f"\n{column}")
    print(validation_groups[column].value_counts(dropna=False))


Gender
Gender
Female    2902
Male      1898
Name: count, dtype: int64

Age
Age
30-39    1802
21-29    1505
40-49    1071
50-59     365
60+        57
Name: count, dtype: int64

Education
Education
University         2279
Graduate School    1656
High School         784
Other                81
Name: count, dtype: int64

Marital Status
Marital Status
Single     2528
Married    2208
Other        64
Name: count, dtype: int64

Gender × Age
Gender × Age
Female × 30-39    1081
Female × 21-29     998
Male × 30-39       721
Female × 40-49     603
Male × 21-29       507
Male × 40-49       468
Female × 50-59     193
Male × 50-59       172
Male × 60+          30
Female × 60+        27
Name: count, dtype: int64


In [23]:
group_fairness_results = []

for (strategy, model_name), predictions in model_predictions.items():

    y_pred = predictions["y_pred"]

    for group_name in validation_groups.columns:

        group_values = validation_groups[group_name]

        for group in group_values.dropna().unique():

            mask = (group_values == group).to_numpy()

            actual = y_val.to_numpy()[mask]
            predicted = np.asarray(y_pred)[mask]

            tn, fp, fn, tp = confusion_matrix(
                actual, predicted, labels=[0, 1]
            ).ravel()

            group_fairness_results.append({
                "Strategy": strategy,
                "Model": model_name,
                "Group Dimension": group_name,
                "Group": group,
                "Count": len(actual),
                "Selection Rate": predicted.mean(),
                "TPR": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
                "FPR": fp / (fp + tn) if (fp + tn) > 0 else np.nan
            })

group_fairness_df = pd.DataFrame(group_fairness_results)

print("Group-level fairness metrics calculated!")
print("Total records:", len(group_fairness_df))

display(group_fairness_df.head(15).round(4))

Group-level fairness metrics calculated!
Total records: 456


,Strategy,Model,Group Dimension,Group,Count,Selection Rate,TPR,FPR
0,Original,Logistic Regression,Gender,Male,1898,0.1054,0.2789,0.0460
1,Original,Logistic Regression,Gender,Female,2902,0.0665,0.2457,0.0219
2,Original,Logistic Regression,Age,21-29,1505,0.0797,0.2394,0.0349
3,Original,Logistic Regression,Age,40-49,1071,0.0952,0.2966,0.0297
4,Original,Logistic Regression,Age,30-39,1802,0.0721,0.2632,0.0243
5,Original,Logistic Regression,Age,60+,57,0.1404,0.2941,0.0750
6,Original,Logistic Regression,Age,50-59,365,0.0904,0.2198,0.0474
7,Original,Logistic Regression,Education,University,2279,0.1009,0.3119,0.0371
8,Original,Logistic Regression,Education,Graduate School,1656,0.0459,0.1662,0.0165
9,Original,Logistic Regression,Education,High School,784,0.1110,0.2886,0.0497


In [24]:
def calculate_gap(values):
    values = pd.Series(values).dropna()

    if len(values) < 2:
        return np.nan

    return values.max() - values.min()


fairness_gap_results = []

for (strategy, model_name), predictions in model_predictions.items():

    for group_name in validation_groups.columns:

        subset = group_fairness_df[
            (group_fairness_df["Strategy"] == strategy) &
            (group_fairness_df["Model"] == model_name) &
            (group_fairness_df["Group Dimension"] == group_name)
        ]

        dp_gap = calculate_gap(subset["Selection Rate"])
        tpr_gap = (
            calculate_gap(subset["TPR"])
            if subset["TPR"].notna().all() else np.nan
        )
        fpr_gap = (
            calculate_gap(subset["FPR"])
            if subset["FPR"].notna().all() else np.nan
        )

        # Favourable outcome = predicted non-default
        favourable_rates = 1 - subset["Selection Rate"]

        if favourable_rates.max() > 0:
            disparate_impact = (
                favourable_rates.min() / favourable_rates.max()
            )
        else:
            disparate_impact = np.nan

        fairness_gap_results.append({
            "Strategy": strategy,
            "Model": model_name,
            "Group Dimension": group_name,
            "Demographic Parity Gap": dp_gap,
            "Equal Opportunity Gap": tpr_gap,
            "FPR Gap": fpr_gap,
            "Equalized Odds Gap": (
                max(tpr_gap, fpr_gap)
                if pd.notna(tpr_gap) and pd.notna(fpr_gap)
                else np.nan
            ),
            "Disparate Impact": disparate_impact
        })


fairness_gaps_df = pd.DataFrame(fairness_gap_results)

print("Fairness gaps calculated!")
print("Total fairness comparisons:", len(fairness_gaps_df))

display(fairness_gaps_df.head(15).round(4))

Fairness gaps calculated!
Total fairness comparisons: 95


,Strategy,Model,Group Dimension,Demographic Parity Gap,Equal Opportunity Gap,FPR Gap,Equalized Odds Gap,Disparate Impact
0,Original,Logistic Regression,Gender,0.0389,0.0333,0.0240,0.0333,0.9584
1,Original,Logistic Regression,Age,0.0682,0.0768,0.0507,0.0768,0.9265
2,Original,Logistic Regression,Education,0.1110,0.3119,0.0497,0.3119,0.8890
3,Original,Logistic Regression,Marital Status,0.0730,0.1744,0.0348,0.1744,0.9217
4,Original,Logistic Regression,Gender × Age,0.1343,0.3800,0.0786,0.3800,0.8585
5,Original,KNN,Gender,0.0355,0.0039,0.0243,0.0243,0.9604
6,Original,KNN,Age,0.0667,0.1700,0.0292,0.1700,0.9252
7,Original,KNN,Education,0.0913,0.1043,0.0364,0.1043,0.9027
8,Original,KNN,Marital Status,0.0439,0.0635,0.0294,0.0635,0.9505
9,Original,KNN,Gender × Age,0.1029,0.2945,0.1429,0.2945,0.8879


In [25]:
calibration_results = []

for (strategy, model_name), predictions in model_predictions.items():

    y_prob = np.asarray(predictions["y_prob"])
    y_actual = y_val.to_numpy()

    for group_name in validation_groups.columns:

        group_values = validation_groups[group_name]

        for group in group_values.dropna().unique():

            mask = (group_values == group).to_numpy()

            actual = y_actual[mask]
            probabilities = y_prob[mask]

            mean_predicted = probabilities.mean()
            actual_rate = actual.mean()

            calibration_results.append({
                "Strategy": strategy,
                "Model": model_name,
                "Group Dimension": group_name,
                "Group": group,
                "Count": len(actual),
                "Mean Predicted Risk": mean_predicted,
                "Actual Default Rate": actual_rate,
                "Calibration Error": abs(mean_predicted - actual_rate),
                "Brier Score": np.mean((probabilities - actual) ** 2)
            })

calibration_df = pd.DataFrame(calibration_results)

print("Calibration evaluation completed!")
print("Total group-level records:", len(calibration_df))

display(calibration_df.head(15).round(4))

Calibration evaluation completed!
Total group-level records: 456


,Strategy,Model,Group Dimension,Group,Count,Mean Predicted Risk,Actual Default Rate,Calibration Error,Brier Score
0,Original,Logistic Regression,Gender,Male,1898,0.2406,0.2550,0.0144,0.1637
1,Original,Logistic Regression,Gender,Female,2902,0.2069,0.1992,0.0077,0.1349
2,Original,Logistic Regression,Age,21-29,1505,0.2260,0.2193,0.0067,0.1458
3,Original,Logistic Regression,Age,40-49,1071,0.2243,0.2456,0.0213,0.1535
4,Original,Logistic Regression,Age,30-39,1802,0.2077,0.2003,0.0073,0.1369
5,Original,Logistic Regression,Age,60+,57,0.2576,0.2982,0.0406,0.2007
6,Original,Logistic Regression,Age,50-59,365,0.2405,0.2493,0.0088,0.1653
7,Original,Logistic Regression,Education,University,2279,0.2392,0.2321,0.0070,0.1489
8,Original,Logistic Regression,Education,Graduate School,1656,0.1899,0.1963,0.0063,0.1412
9,Original,Logistic Regression,Education,High School,784,0.2448,0.2564,0.0115,0.1567


In [26]:
calibration_gap_results = []

for (strategy, model_name) in model_predictions:

    for group_name in validation_groups.columns:

        subset = calibration_df[
            (calibration_df["Strategy"] == strategy) &
            (calibration_df["Model"] == model_name) &
            (calibration_df["Group Dimension"] == group_name)
        ]

        calibration_gap_results.append({
            "Strategy": strategy,
            "Model": model_name,
            "Group Dimension": group_name,
            "Calibration Error Gap": calculate_gap(
                subset["Calibration Error"]
            ),
            "Brier Score Gap": calculate_gap(
                subset["Brier Score"]
            )
        })

calibration_gaps_df = pd.DataFrame(calibration_gap_results)

print("Calibration gaps calculated!")
display(calibration_gaps_df.head(15).round(4))

Calibration gaps calculated!


,Strategy,Model,Group Dimension,Calibration Error Gap,Brier Score Gap
0,Original,Logistic Regression,Gender,0.0067,0.0289
1,Original,Logistic Regression,Age,0.0339,0.0638
2,Original,Logistic Regression,Education,0.0124,0.0811
3,Original,Logistic Regression,Marital Status,0.0313,0.0141
4,Original,Logistic Regression,Gender × Age,0.1294,0.1143
5,Original,KNN,Gender,0.0160,0.0299
6,Original,KNN,Age,0.0534,0.0467
7,Original,KNN,Education,0.0661,0.0646
8,Original,KNN,Marital Status,0.0191,0.0145
9,Original,KNN,Gender × Age,0.1433,0.0962


In [27]:
counterfactual_results = []

# Create validation data with gender flipped
X_val_counterfactual = X_val.copy()

X_val_counterfactual["SEX"] = X_val_counterfactual["SEX"].map({
    1: 2,
    2: 1
})

# Transform counterfactual data using the existing preprocessor
X_counterfactual_processed = preprocessor.transform(
    X_val_counterfactual
)

for (strategy, model_name), model in trained_models.items():

    original_prob = model.predict_proba(X_val_processed)[:, 1]
    counterfactual_prob = model.predict_proba(
        X_counterfactual_processed
    )[:, 1]

    original_pred = model.predict(X_val_processed)
    counterfactual_pred = model.predict(
        X_counterfactual_processed
    )

    counterfactual_results.append({
        "Strategy": strategy,
        "Model": model_name,
        "Mean Probability Change": np.mean(
            np.abs(original_prob - counterfactual_prob)
        ),
        "Maximum Probability Change": np.max(
            np.abs(original_prob - counterfactual_prob)
        ),
        "Prediction Flip Rate": np.mean(
            original_pred != counterfactual_pred
        )
    })

counterfactual_df = pd.DataFrame(counterfactual_results)

print("Counterfactual sensitivity test completed!")
display(counterfactual_df.round(4))

Counterfactual sensitivity test completed!


,Strategy,Model,Mean Probability Change,Maximum Probability Change,Prediction Flip Rate
0,Original,Logistic Regression,0.0136,0.0231,0.0121
1,Original,KNN,0.0768,0.4286,0.0350
2,Original,Naive Bayes,0.0027,0.1001,0.0021
3,Original,SVM,0.0054,0.1160,0.0023
4,Original,Decision Tree,0.0314,1.0000,0.0317
5,Original,Random Forest,0.0332,0.6400,0.0275
6,Original,XGBoost,0.0185,0.3395,0.0165
7,Class Weight,Logistic Regression,0.0198,0.0231,0.0729
8,Class Weight,SVM,0.0099,0.1072,0.0094
9,Class Weight,Decision Tree,0.0232,1.0000,0.0231


In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# Exclude protected attributes when measuring similarity
protected_columns = ["SEX", "AGE", "EDUCATION", "MARRIAGE"]

X_train_similarity = X_train.drop(columns=protected_columns)
X_val_similarity = X_val.drop(columns=protected_columns)

# Fit scaling only on training data
similarity_scaler = StandardScaler()

X_train_similarity_scaled = similarity_scaler.fit_transform(
    X_train_similarity
)

X_val_similarity_scaled = similarity_scaler.transform(
    X_val_similarity
)

# Find the nearest other applicant for every validation applicant
nearest_neighbors = NearestNeighbors(n_neighbors=2, metric="euclidean")

nearest_neighbors.fit(X_val_similarity_scaled)

distances, indices = nearest_neighbors.kneighbors(
    X_val_similarity_scaled
)

# First neighbor is the applicant themself; second is the closest other applicant
nearest_indices = indices[:, 1]
nearest_distances = distances[:, 1]

individual_fairness_results = []

for (strategy, model_name), predictions in model_predictions.items():

    probabilities = np.asarray(predictions["y_prob"])
    predicted_labels = np.asarray(predictions["y_pred"])

    probability_differences = np.abs(
        probabilities - probabilities[nearest_indices]
    )

    prediction_disagreements = (
        predicted_labels != predicted_labels[nearest_indices]
    )

    individual_fairness_results.append({
        "Strategy": strategy,
        "Model": model_name,
        "Mean Probability Difference": probability_differences.mean(),
        "Maximum Probability Difference": probability_differences.max(),
        "Prediction Disagreement Rate": prediction_disagreements.mean(),
        "Mean Nearest Neighbor Distance": nearest_distances.mean()
    })

individual_fairness_df = pd.DataFrame(individual_fairness_results)

print("Individual fairness evaluation completed!")
display(individual_fairness_df.round(4))

Individual fairness evaluation completed!


,Strategy,Model,Mean Probability Difference,Maximum Probability Difference,Prediction Disagreement Rate,Mean Nearest Neighbor Distance
0,Original,Logistic Regression,0.0307,0.4208,0.0246,0.7601
1,Original,KNN,0.0829,0.4762,0.0412,0.7601
2,Original,Naive Bayes,0.0483,1.0000,0.0471,0.7601
3,Original,SVM,0.0136,0.5107,0.0108,0.7601
4,Original,Decision Tree,0.2722,1.0000,0.2719,0.7601
5,Original,Random Forest,0.0799,0.7400,0.0502,0.7601
6,Original,XGBoost,0.0940,0.9667,0.0694,0.7601
7,Class Weight,Logistic Regression,0.0457,0.5187,0.1369,0.7601
8,Class Weight,SVM,0.0190,0.3698,0.0258,0.7601
9,Class Weight,Decision Tree,0.2738,1.0000,0.2742,0.7601


In [29]:
# 1. Demographic Parity and Equal Opportunity
dp_eo_summary = fairness_gaps_df.groupby(
    ["Strategy", "Model"]
).agg(
    Max_DP_Gap=("Demographic Parity Gap", "max"),
    Max_EO_Gap=("Equal Opportunity Gap", "max"),
    Min_Disparate_Impact=("Disparate Impact", "min")
).reset_index()


# 2. Calibration Bias
calibration_summary = calibration_gaps_df.groupby(
    ["Strategy", "Model"]
).agg(
    Max_Calibration_Error_Gap=("Calibration Error Gap", "max")
).reset_index()


# 3. Intersectional Bias: Gender × Age
intersectional_summary = fairness_gaps_df[
    fairness_gaps_df["Group Dimension"] == "Gender × Age"
][[
    "Strategy", "Model",
    "Demographic Parity Gap",
    "Equal Opportunity Gap"
]].rename(columns={
    "Demographic Parity Gap": "Intersectional_DP_Gap",
    "Equal Opportunity Gap": "Intersectional_EO_Gap"
})


# 4. Counterfactual Fairness sensitivity
counterfactual_summary = counterfactual_df[[
    "Strategy", "Model",
    "Mean Probability Change",
    "Prediction Flip Rate"
]]


# 5. Individual Fairness
individual_summary = individual_fairness_df[[
    "Strategy", "Model",
    "Mean Probability Difference",
    "Prediction Disagreement Rate"
]]


# 6. Combine all dimensions
fairness_summary = (
    dp_eo_summary
    .merge(calibration_summary, on=["Strategy", "Model"])
    .merge(intersectional_summary, on=["Strategy", "Model"])
    .merge(counterfactual_summary, on=["Strategy", "Model"])
    .merge(individual_summary, on=["Strategy", "Model"])
)

print("Seven fairness dimensions combined!")
print("Total configurations:", len(fairness_summary))

display(fairness_summary.round(4))

Seven fairness dimensions combined!
Total configurations: 19


,Strategy,Model,Max_DP_Gap,Max_EO_Gap,Min_Disparate_Impact,Max_Calibration_Error_Gap,Intersectional_DP_Gap,Intersectional_EO_Gap,Mean Probability Change,Prediction Flip Rate,Mean Probability Difference,Prediction Disagreement Rate
0,Class Weight,Decision Tree,0.1531,0.4064,0.8132,0.1107,0.1531,0.1945,0.0232,0.0231,0.2738,0.2742
1,Class Weight,Logistic Regression,0.4358,0.6183,0.5532,0.1736,0.3058,0.2674,0.0198,0.0729,0.0457,0.1369
2,Class Weight,Random Forest,0.2032,0.2566,0.7863,0.1478,0.1325,0.2255,0.0411,0.0369,0.0944,0.0796
3,Class Weight,SVM,0.2363,0.3511,0.7414,0.1615,0.1704,0.2121,0.0099,0.0094,0.0190,0.0258
4,Class Weight,XGBoost,0.2865,0.5970,0.7062,0.1366,0.1863,0.2467,0.0285,0.0298,0.1332,0.1435
5,Original,Decision Tree,0.2502,0.4030,0.6895,0.2137,0.2502,0.2528,0.0314,0.0317,0.2722,0.2719
6,Original,KNN,0.1029,0.2945,0.8879,0.1433,0.1029,0.2945,0.0768,0.0350,0.0829,0.0412
7,Original,Logistic Regression,0.1343,0.3800,0.8585,0.1294,0.1343,0.3800,0.0136,0.0121,0.0307,0.0246
8,Original,Naive Bayes,0.9487,0.9792,0.0513,0.6272,0.0762,0.0909,0.0027,0.0021,0.0483,0.0471
9,Original,Random Forest,0.1538,0.3255,0.8329,0.0882,0.1538,0.3255,0.0332,0.0275,0.0799,0.0502


In [30]:
# Save all pre-mitigation fairness results
group_fairness_df.to_csv(
    results_path / "stage5_group_fairness_before.csv", index=False
)

fairness_gaps_df.to_csv(
    results_path / "stage5_fairness_gaps_before.csv", index=False
)

calibration_df.to_csv(
    results_path / "stage5_calibration_before.csv", index=False
)

calibration_gaps_df.to_csv(
    results_path / "stage5_calibration_gaps_before.csv", index=False
)

counterfactual_df.to_csv(
    results_path / "stage5_counterfactual_before.csv", index=False
)

individual_fairness_df.to_csv(
    results_path / "stage5_individual_fairness_before.csv", index=False
)

fairness_summary.to_csv(
    results_path / "stage5_fairness_summary_before.csv", index=False
)

print("All pre-mitigation fairness results saved!")
print("Configurations:", len(fairness_summary))
print("Group-level records:", len(group_fairness_df))
print("Fairness gap records:", len(fairness_gaps_df))
print("Calibration records:", len(calibration_df))

All pre-mitigation fairness results saved!
Configurations: 19
Group-level records: 456
Fairness gap records: 95
Calibration records: 456


In [31]:
# Remove protected attributes from training and validation features
protected_columns = ["SEX", "AGE", "EDUCATION", "MARRIAGE"]

X_train_unaware = X_train.drop(columns=protected_columns)
X_val_unaware = X_val.drop(columns=protected_columns)
X_train_smote_unaware = X_train_smote.drop(columns=protected_columns)

print("Protected attributes removed!")
print("Training shape:", X_train_unaware.shape)
print("Validation shape:", X_val_unaware.shape)
print("SMOTENC training shape:", X_train_smote_unaware.shape)

Protected attributes removed!
Training shape: (19200, 19)
Validation shape: (4800, 19)
SMOTENC training shape: (29906, 19)


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_columns_unaware = [
    col for col in ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
    if col in X_train_unaware.columns
]

numerical_columns_unaware = [
    col for col in X_train_unaware.columns
    if col not in categorical_columns_unaware
]

preprocessor_unaware = ColumnTransformer([
    ("numerical", StandardScaler(), numerical_columns_unaware),
    ("ordinal", "passthrough", categorical_columns_unaware)
])

X_train_unaware_processed = preprocessor_unaware.fit_transform(X_train_unaware)
X_val_unaware_processed = preprocessor_unaware.transform(X_val_unaware)
X_train_smote_unaware_processed = preprocessor_unaware.transform(
    X_train_smote_unaware
)

print("Preprocessing completed!")
print("Original:", X_train_unaware_processed.shape)
print("Validation:", X_val_unaware_processed.shape)
print("SMOTENC:", X_train_smote_unaware_processed.shape)

Preprocessing completed!
Original: (19200, 19)
Validation: (4800, 19)
SMOTENC: (29906, 19)


In [33]:
strategy_data_unaware = {
    "Original": (X_train_unaware_processed, y_train),
    "Class Weight": (X_train_unaware_processed, y_train),
    "SMOTENC": (X_train_smote_unaware_processed, y_train_smote)
}

sensitive_removed_results = []
sensitive_removed_predictions = {}
sensitive_removed_models = {}

for strategy, (X_data, y_data) in strategy_data_unaware.items():
    for model_name in model_names:

        if strategy == "Class Weight":
            if model_name not in class_weight_models:
                continue
            model = clone(class_weight_models[model_name])
        else:
            model = clone(models[model_name])

        model.fit(X_data, y_data)

        y_pred = model.predict(X_val_unaware_processed)
        y_prob = model.predict_proba(X_val_unaware_processed)[:, 1]

        tn, fp, fn, tp = confusion_matrix(
            y_val, y_pred, labels=[0, 1]
        ).ravel()

        sensitive_removed_results.append({
            "Strategy": strategy,
            "Model": model_name,
            "Accuracy": accuracy_score(y_val, y_pred),
            "Precision": precision_score(y_val, y_pred, zero_division=0),
            "Recall": recall_score(y_val, y_pred, zero_division=0),
            "F1": f1_score(y_val, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_val, y_prob),
            "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
            "FNR": fn / (fn + tp) if (fn + tp) else np.nan
        })

        sensitive_removed_predictions[(strategy, model_name)] = {
            "y_pred": y_pred,
            "y_prob": y_prob
        }

        sensitive_removed_models[(strategy, model_name)] = model

sensitive_removed_df = pd.DataFrame(sensitive_removed_results)

print("Sensitive-attribute-removed models trained!")
print("Configurations:", len(sensitive_removed_df))

display(sensitive_removed_df.round(4))

Sensitive-attribute-removed models trained!
Configurations: 19


,Strategy,Model,Accuracy,Precision,Recall,F1,ROC-AUC,FPR,FNR
0,Original,Logistic Regression,0.8112,0.7120,0.2467,0.3664,0.7184,0.0284,0.7533
1,Original,KNN,0.8171,0.6625,0.3531,0.4607,0.7531,0.0511,0.6469
2,Original,Naive Bayes,0.7698,0.4832,0.5810,0.5276,0.7347,0.1766,0.4190
3,Original,SVM,0.8200,0.6861,0.3437,0.4580,0.7128,0.0447,0.6563
4,Original,Decision Tree,0.7244,0.3804,0.3908,0.3855,0.6131,0.1808,0.6092
5,Original,Random Forest,0.8131,0.6333,0.3691,0.4664,0.7625,0.0607,0.6309
6,Original,XGBoost,0.8133,0.6402,0.3569,0.4583,0.7555,0.0570,0.6431
7,Class Weight,Logistic Regression,0.6844,0.3771,0.6544,0.4785,0.7191,0.3071,0.3456
8,Class Weight,SVM,0.7696,0.4828,0.5829,0.5282,0.7569,0.1774,0.4171
9,Class Weight,Decision Tree,0.7254,0.3859,0.4077,0.3965,0.6124,0.1843,0.5923


In [34]:
sensitive_removed_df.to_csv(
    results_path / "stage5_sensitive_removed_validation.csv",
    index=False
)

print("Sensitive-attribute-removal results saved!")

Sensitive-attribute-removal results saved!


In [35]:
# Evaluate group-level fairness after sensitive attribute removal
sensitive_removed_group_results = []

for (strategy, model_name), predictions in sensitive_removed_predictions.items():
    y_pred = predictions["y_pred"]

    for group_name in validation_groups.columns:
        group_values = validation_groups[group_name]

        for group in group_values.dropna().unique():
            mask = (group_values == group).to_numpy()
            actual = y_val.to_numpy()[mask]
            predicted = np.asarray(y_pred)[mask]

            tn, fp, fn, tp = confusion_matrix(
                actual, predicted, labels=[0, 1]
            ).ravel()

            sensitive_removed_group_results.append({
                "Strategy": strategy,
                "Model": model_name,
                "Group Dimension": group_name,
                "Group": group,
                "Count": len(actual),
                "Selection Rate": predicted.mean(),
                "TPR": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
                "FPR": fp / (fp + tn) if (fp + tn) > 0 else np.nan
            })

sensitive_removed_group_df = pd.DataFrame(
    sensitive_removed_group_results
)

print("Group-level fairness evaluation completed!")
print("Records:", len(sensitive_removed_group_df))
display(sensitive_removed_group_df.head(10).round(4))

Group-level fairness evaluation completed!
Records: 456


,Strategy,Model,Group Dimension,Group,Count,Selection Rate,TPR,FPR
0,Original,Logistic Regression,Gender,Male,1898,0.0917,0.2438,0.0396
1,Original,Logistic Regression,Gender,Female,2902,0.0669,0.2491,0.0215
2,Original,Logistic Regression,Age,21-29,1505,0.0864,0.2606,0.0374
3,Original,Logistic Regression,Age,40-49,1071,0.0812,0.2662,0.0210
4,Original,Logistic Regression,Age,30-39,1802,0.0660,0.2327,0.0243
5,Original,Logistic Regression,Age,60+,57,0.1053,0.2353,0.0500
6,Original,Logistic Regression,Age,50-59,365,0.0712,0.1978,0.0292
7,Original,Logistic Regression,Education,University,2279,0.0970,0.2968,0.0366
8,Original,Logistic Regression,Education,Graduate School,1656,0.0393,0.1477,0.0128
9,Original,Logistic Regression,Education,High School,784,0.1033,0.2836,0.0412


In [36]:
sensitive_removed_gap_results = []

for (strategy, model_name) in sensitive_removed_predictions:
    for group_name in validation_groups.columns:

        subset = sensitive_removed_group_df[
            (sensitive_removed_group_df["Strategy"] == strategy) &
            (sensitive_removed_group_df["Model"] == model_name) &
            (sensitive_removed_group_df["Group Dimension"] == group_name)
        ]

        dp_gap = calculate_gap(subset["Selection Rate"])

        tpr_gap = (
            calculate_gap(subset["TPR"])
            if subset["TPR"].notna().all() else np.nan
        )

        fpr_gap = (
            calculate_gap(subset["FPR"])
            if subset["FPR"].notna().all() else np.nan
        )

        favourable_rates = 1 - subset["Selection Rate"]

        disparate_impact = (
            favourable_rates.min() / favourable_rates.max()
            if favourable_rates.max() > 0 else np.nan
        )

        sensitive_removed_gap_results.append({
            "Strategy": strategy,
            "Model": model_name,
            "Group Dimension": group_name,
            "Demographic Parity Gap": dp_gap,
            "Equal Opportunity Gap": tpr_gap,
            "FPR Gap": fpr_gap,
            "Equalized Odds Gap": max(tpr_gap, fpr_gap)
                if pd.notna(tpr_gap) and pd.notna(fpr_gap)
                else np.nan,
            "Disparate Impact": disparate_impact
        })

sensitive_removed_gaps_df = pd.DataFrame(
    sensitive_removed_gap_results
)

print("Fairness gaps calculated!")
print("Records:", len(sensitive_removed_gaps_df))
display(sensitive_removed_gaps_df.head(10).round(4))

Fairness gaps calculated!
Records: 95


,Strategy,Model,Group Dimension,Demographic Parity Gap,Equal Opportunity Gap,FPR Gap,Equalized Odds Gap,Disparate Impact
0,Original,Logistic Regression,Gender,0.0248,0.0053,0.0181,0.0181,0.9734
1,Original,Logistic Regression,Age,0.0392,0.0684,0.0290,0.0684,0.9580
2,Original,Logistic Regression,Education,0.0910,0.2968,0.0284,0.2968,0.9079
3,Original,Logistic Regression,Marital Status,0.0506,0.1576,0.0141,0.1576,0.9453
4,Original,Logistic Regression,Gender × Age,0.1519,0.4091,0.0952,0.4091,0.8429
5,Original,KNN,Gender,0.0350,0.0072,0.0259,0.0259,0.9609
6,Original,KNN,Age,0.0328,0.1321,0.0388,0.1321,0.9634
7,Original,KNN,Education,0.1015,0.1538,0.0491,0.1538,0.8918
8,Original,KNN,Marital Status,0.0411,0.0527,0.0323,0.0527,0.9535
9,Original,KNN,Gender × Age,0.0815,0.3200,0.0791,0.3200,0.9127


In [37]:
sensitive_removed_calibration_results = []

for (strategy, model_name), predictions in sensitive_removed_predictions.items():
    y_prob = np.asarray(predictions["y_prob"])
    y_actual = y_val.to_numpy()

    for group_name in validation_groups.columns:
        group_values = validation_groups[group_name]

        for group in group_values.dropna().unique():
            mask = (group_values == group).to_numpy()

            actual = y_actual[mask]
            probabilities = y_prob[mask]

            mean_predicted = probabilities.mean()
            actual_rate = actual.mean()

            sensitive_removed_calibration_results.append({
                "Strategy": strategy,
                "Model": model_name,
                "Group Dimension": group_name,
                "Group": group,
                "Count": len(actual),
                "Mean Predicted Risk": mean_predicted,
                "Actual Default Rate": actual_rate,
                "Calibration Error": abs(mean_predicted - actual_rate),
                "Brier Score": np.mean((probabilities - actual) ** 2)
            })

sensitive_removed_calibration_df = pd.DataFrame(
    sensitive_removed_calibration_results
)

print("Calibration evaluation completed!")
print("Records:", len(sensitive_removed_calibration_df))
display(sensitive_removed_calibration_df.head(10).round(4))

Calibration evaluation completed!
Records: 456


,Strategy,Model,Group Dimension,Group,Count,Mean Predicted Risk,Actual Default Rate,Calibration Error,Brier Score
0,Original,Logistic Regression,Gender,Male,1898,0.2317,0.2550,0.0234,0.1657
1,Original,Logistic Regression,Gender,Female,2902,0.2127,0.1992,0.0135,0.1352
2,Original,Logistic Regression,Age,21-29,1505,0.2399,0.2193,0.0206,0.1461
3,Original,Logistic Regression,Age,40-49,1071,0.2132,0.2456,0.0323,0.1548
4,Original,Logistic Regression,Age,30-39,1802,0.2070,0.2003,0.0066,0.1378
5,Original,Logistic Regression,Age,60+,57,0.2324,0.2982,0.0658,0.2055
6,Original,Logistic Regression,Age,50-59,365,0.2228,0.2493,0.0265,0.1680
7,Original,Logistic Regression,Education,University,2279,0.2385,0.2321,0.0064,0.1498
8,Original,Logistic Regression,Education,Graduate School,1656,0.1836,0.1963,0.0126,0.1420
9,Original,Logistic Regression,Education,High School,784,0.2495,0.2564,0.0069,0.1572


In [38]:
sensitive_removed_calibration_gaps = []

for (strategy, model_name) in sensitive_removed_predictions:
    for group_name in validation_groups.columns:

        subset = sensitive_removed_calibration_df[
            (sensitive_removed_calibration_df["Strategy"] == strategy) &
            (sensitive_removed_calibration_df["Model"] == model_name) &
            (sensitive_removed_calibration_df["Group Dimension"] == group_name)
        ]

        sensitive_removed_calibration_gaps.append({
            "Strategy": strategy,
            "Model": model_name,
            "Group Dimension": group_name,
            "Calibration Error Gap": calculate_gap(
                subset["Calibration Error"]
            ),
            "Brier Score Gap": calculate_gap(
                subset["Brier Score"]
            )
        })

sensitive_removed_calibration_gaps_df = pd.DataFrame(
    sensitive_removed_calibration_gaps
)

print("Calibration gaps calculated!")
print("Records:", len(sensitive_removed_calibration_gaps_df))
display(sensitive_removed_calibration_gaps_df.head(10).round(4))

Calibration gaps calculated!
Records: 95


,Strategy,Model,Group Dimension,Calibration Error Gap,Brier Score Gap
0,Original,Logistic Regression,Gender,0.0098,0.0304
1,Original,Logistic Regression,Age,0.0592,0.0678
2,Original,Logistic Regression,Education,0.0781,0.0687
3,Original,Logistic Regression,Marital Status,0.0169,0.0192
4,Original,Logistic Regression,Gender × Age,0.1615,0.1224
5,Original,KNN,Gender,0.0071,0.0331
6,Original,KNN,Age,0.0366,0.0516
7,Original,KNN,Education,0.0666,0.0565
8,Original,KNN,Marital Status,0.0080,0.0118
9,Original,KNN,Gender × Age,0.1308,0.0862


In [39]:
# Create gender-swapped validation data
X_val_counterfactual_unaware = X_val.copy()

X_val_counterfactual_unaware["SEX"] = (
    X_val_counterfactual_unaware["SEX"].map({1: 2, 2: 1})
)

# Remove protected attributes before prediction
X_val_counterfactual_unaware = (
    X_val_counterfactual_unaware.drop(columns=protected_columns)
)

X_counterfactual_unaware_processed = (
    preprocessor_unaware.transform(X_val_counterfactual_unaware)
)

sensitive_removed_counterfactual_results = []

for (strategy, model_name), model in sensitive_removed_models.items():

    original_prob = model.predict_proba(
        X_val_unaware_processed
    )[:, 1]

    counterfactual_prob = model.predict_proba(
        X_counterfactual_unaware_processed
    )[:, 1]

    original_pred = model.predict(X_val_unaware_processed)
    counterfactual_pred = model.predict(
        X_counterfactual_unaware_processed
    )

    sensitive_removed_counterfactual_results.append({
        "Strategy": strategy,
        "Model": model_name,
        "Mean Probability Change": np.mean(
            np.abs(original_prob - counterfactual_prob)
        ),
        "Maximum Probability Change": np.max(
            np.abs(original_prob - counterfactual_prob)
        ),
        "Prediction Flip Rate": np.mean(
            original_pred != counterfactual_pred
        )
    })

sensitive_removed_counterfactual_df = pd.DataFrame(
    sensitive_removed_counterfactual_results
)

print("Counterfactual sensitivity evaluation completed!")
display(sensitive_removed_counterfactual_df.round(4))

Counterfactual sensitivity evaluation completed!


,Strategy,Model,Mean Probability Change,Maximum Probability Change,Prediction Flip Rate
0,Original,Logistic Regression,0.0,0.0,0.0
1,Original,KNN,0.0,0.0,0.0
2,Original,Naive Bayes,0.0,0.0,0.0
3,Original,SVM,0.0,0.0,0.0
4,Original,Decision Tree,0.0,0.0,0.0
5,Original,Random Forest,0.0,0.0,0.0
6,Original,XGBoost,0.0,0.0,0.0
7,Class Weight,Logistic Regression,0.0,0.0,0.0
8,Class Weight,SVM,0.0,0.0,0.0
9,Class Weight,Decision Tree,0.0,0.0,0.0


In [40]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# Use non-protected features to identify similar applicants
similarity_scaler = StandardScaler()

X_similarity_train = similarity_scaler.fit_transform(
    X_train_unaware
)

X_similarity_val = similarity_scaler.transform(
    X_val_unaware
)

# Find the nearest other applicant
nearest_neighbors = NearestNeighbors(n_neighbors=2)
nearest_neighbors.fit(X_similarity_val)

distances, neighbor_indices = nearest_neighbors.kneighbors(
    X_similarity_val
)

nearest_indices = neighbor_indices[:, 1]

sensitive_removed_individual_results = []

for (strategy, model_name), predictions in sensitive_removed_predictions.items():

    probabilities = np.asarray(predictions["y_prob"])
    predicted_labels = np.asarray(predictions["y_pred"])

    probability_differences = np.abs(
        probabilities - probabilities[nearest_indices]
    )

    prediction_disagreement = (
        predicted_labels != predicted_labels[nearest_indices]
    )

    sensitive_removed_individual_results.append({
        "Strategy": strategy,
        "Model": model_name,
        "Mean Probability Difference": probability_differences.mean(),
        "Maximum Probability Difference": probability_differences.max(),
        "Prediction Disagreement Rate": prediction_disagreement.mean()
    })

sensitive_removed_individual_df = pd.DataFrame(
    sensitive_removed_individual_results
)

print("Individual fairness evaluation completed!")
display(sensitive_removed_individual_df.round(4))

Individual fairness evaluation completed!


,Strategy,Model,Mean Probability Difference,Maximum Probability Difference,Prediction Disagreement Rate
0,Original,Logistic Regression,0.0137,0.3767,0.0127
1,Original,KNN,0.0471,0.5714,0.0206
2,Original,Naive Bayes,0.0199,0.8483,0.0119
3,Original,SVM,0.0087,0.5019,0.0096
4,Original,Decision Tree,0.2603,1.0000,0.2594
5,Original,Random Forest,0.0840,0.9000,0.0577
6,Original,XGBoost,0.0874,0.9319,0.0612
7,Class Weight,Logistic Regression,0.0199,0.4607,0.0454
8,Class Weight,SVM,0.0104,0.3409,0.0150
9,Class Weight,Decision Tree,0.2436,1.0000,0.2448


In [41]:
# 1. Demographic Parity and Equal Opportunity
dp_eo_unaware = sensitive_removed_gaps_df.groupby(
    ["Strategy", "Model"]
).agg(
    Max_DP_Gap=("Demographic Parity Gap", "max"),
    Max_EO_Gap=("Equal Opportunity Gap", "max"),
    Min_Disparate_Impact=("Disparate Impact", "min")
).reset_index()

# 2. Calibration Bias
calibration_unaware = sensitive_removed_calibration_gaps_df.groupby(
    ["Strategy", "Model"]
).agg(
    Max_Calibration_Error_Gap=("Calibration Error Gap", "max")
).reset_index()

# 3. Intersectional Bias
intersectional_unaware = sensitive_removed_gaps_df[
    sensitive_removed_gaps_df["Group Dimension"] == "Gender × Age"
][[
    "Strategy", "Model",
    "Demographic Parity Gap",
    "Equal Opportunity Gap"
]].rename(columns={
    "Demographic Parity Gap": "Intersectional_DP_Gap",
    "Equal Opportunity Gap": "Intersectional_EO_Gap"
})

# 4. Counterfactual Fairness sensitivity
counterfactual_unaware = sensitive_removed_counterfactual_df[[
    "Strategy", "Model",
    "Mean Probability Change",
    "Prediction Flip Rate"
]]

# 5. Individual Fairness
individual_unaware = sensitive_removed_individual_df[[
    "Strategy", "Model",
    "Mean Probability Difference",
    "Prediction Disagreement Rate"
]]

# 6. Combine all dimensions
sensitive_removed_fairness_summary = (
    dp_eo_unaware
    .merge(calibration_unaware, on=["Strategy", "Model"])
    .merge(intersectional_unaware, on=["Strategy", "Model"])
    .merge(counterfactual_unaware, on=["Strategy", "Model"])
    .merge(individual_unaware, on=["Strategy", "Model"])
)

print("All seven fairness dimensions combined!")
print("Configurations:", len(sensitive_removed_fairness_summary))

display(sensitive_removed_fairness_summary.round(4))

All seven fairness dimensions combined!
Configurations: 19


,Strategy,Model,Max_DP_Gap,Max_EO_Gap,Min_Disparate_Impact,Max_Calibration_Error_Gap,Intersectional_DP_Gap,Intersectional_EO_Gap,Mean Probability Change,Prediction Flip Rate,Mean Probability Difference,Prediction Disagreement Rate
0,Class Weight,Decision Tree,0.1027,0.1670,0.8759,0.1938,0.0634,0.1304,0.0,0.0,0.2436,0.2448
1,Class Weight,Logistic Regression,0.2561,0.2879,0.6707,0.2155,0.2262,0.2879,0.0,0.0,0.0199,0.0454
2,Class Weight,Random Forest,0.1667,0.3267,0.8000,0.1342,0.1667,0.3267,0.0,0.0,0.0989,0.0879
3,Class Weight,SVM,0.2231,0.3710,0.7490,0.1643,0.1333,0.2121,0.0,0.0,0.0104,0.0150
4,Class Weight,XGBoost,0.1801,0.2294,0.7855,0.1635,0.1400,0.1867,0.0,0.0,0.1281,0.1475
5,Original,Decision Tree,0.0768,0.1631,0.9016,0.1318,0.0572,0.1057,0.0,0.0,0.2603,0.2594
6,Original,KNN,0.1015,0.3200,0.8918,0.1308,0.0815,0.3200,0.0,0.0,0.0471,0.0206
7,Original,Logistic Regression,0.1519,0.4091,0.8429,0.1615,0.1519,0.4091,0.0,0.0,0.0137,0.0127
8,Original,Naive Bayes,0.2788,0.3511,0.7028,0.1939,0.2407,0.3030,0.0,0.0,0.0199,0.0119
9,Original,Random Forest,0.1028,0.4848,0.8904,0.1688,0.0893,0.4848,0.0,0.0,0.0840,0.0577


In [42]:
# Save all sensitive-attribute-removal fairness results

sensitive_removed_group_df.to_csv(
    results_path / "stage5_group_fairness_after.csv", index=False
)

sensitive_removed_gaps_df.to_csv(
    results_path / "stage5_fairness_gaps_after.csv", index=False
)

sensitive_removed_calibration_df.to_csv(
    results_path / "stage5_calibration_after.csv", index=False
)

sensitive_removed_calibration_gaps_df.to_csv(
    results_path / "stage5_calibration_gaps_after.csv", index=False
)

sensitive_removed_counterfactual_df.to_csv(
    results_path / "stage5_counterfactual_after.csv", index=False
)

sensitive_removed_individual_df.to_csv(
    results_path / "stage5_individual_fairness_after.csv", index=False
)

sensitive_removed_fairness_summary.to_csv(
    results_path / "stage5_fairness_summary_after.csv", index=False
)

print("Post-mitigation fairness audit saved!")
print("Configurations:", len(sensitive_removed_fairness_summary))

Post-mitigation fairness audit saved!
Configurations: 19


In [43]:
# Compare fairness before and after sensitive attribute removal

comparison_df = fairness_summary.merge(
    sensitive_removed_fairness_summary,
    on=["Strategy", "Model"],
    suffixes=("_Before", "_After")
)

# Calculate changes in fairness metrics
comparison_df["DP Improvement"] = (
    comparison_df["Max_DP_Gap_Before"] -
    comparison_df["Max_DP_Gap_After"]
)

comparison_df["EO Improvement"] = (
    comparison_df["Max_EO_Gap_Before"] -
    comparison_df["Max_EO_Gap_After"]
)

comparison_df["Calibration Improvement"] = (
    comparison_df["Max_Calibration_Error_Gap_Before"] -
    comparison_df["Max_Calibration_Error_Gap_After"]
)

comparison_df["DI Improvement"] = (
    comparison_df["Min_Disparate_Impact_After"] -
    comparison_df["Min_Disparate_Impact_Before"]
)

print("Before vs after comparison completed!")
display(comparison_df[[
    "Strategy", "Model",
    "Max_DP_Gap_Before", "Max_DP_Gap_After", "DP Improvement",
    "Max_EO_Gap_Before", "Max_EO_Gap_After", "EO Improvement",
    "Min_Disparate_Impact_Before", "Min_Disparate_Impact_After",
    "DI Improvement",
    "Max_Calibration_Error_Gap_Before",
    "Max_Calibration_Error_Gap_After",
    "Calibration Improvement"
]].round(4))

Before vs after comparison completed!


,Strategy,Model,Max_DP_Gap_Before,Max_DP_Gap_After,DP Improvement,Max_EO_Gap_Before,Max_EO_Gap_After,EO Improvement,Min_Disparate_Impact_Before,Min_Disparate_Impact_After,DI Improvement,Max_Calibration_Error_Gap_Before,Max_Calibration_Error_Gap_After,Calibration Improvement
0,Class Weight,Decision Tree,0.1531,0.1027,0.0504,0.4064,0.1670,0.2394,0.8132,0.8759,0.0626,0.1107,0.1938,-0.0831
1,Class Weight,Logistic Regression,0.4358,0.2561,0.1797,0.6183,0.2879,0.3305,0.5532,0.6707,0.1175,0.1736,0.2155,-0.0419
2,Class Weight,Random Forest,0.2032,0.1667,0.0365,0.2566,0.3267,-0.0701,0.7863,0.8000,0.0137,0.1478,0.1342,0.0136
3,Class Weight,SVM,0.2363,0.2231,0.0132,0.3511,0.3710,-0.0199,0.7414,0.7490,0.0077,0.1615,0.1643,-0.0027
4,Class Weight,XGBoost,0.2865,0.1801,0.1065,0.5970,0.2294,0.3676,0.7062,0.7855,0.0793,0.1366,0.1635,-0.0269
5,Original,Decision Tree,0.2502,0.0768,0.1734,0.4030,0.1631,0.2399,0.6895,0.9016,0.2121,0.2137,0.1318,0.0819
6,Original,KNN,0.1029,0.1015,0.0013,0.2945,0.3200,-0.0255,0.8879,0.8918,0.0039,0.1433,0.1308,0.0125
7,Original,Logistic Regression,0.1343,0.1519,-0.0175,0.3800,0.4091,-0.0291,0.8585,0.8429,-0.0156,0.1294,0.1615,-0.0321
8,Original,Naive Bayes,0.9487,0.2788,0.6698,0.9792,0.3511,0.6281,0.0513,0.7028,0.6515,0.6272,0.1939,0.4333
9,Original,Random Forest,0.1538,0.1028,0.0510,0.3255,0.4848,-0.1594,0.8329,0.8904,0.0575,0.0882,0.1688,-0.0806


In [44]:
comparison_df.to_csv(
    results_path / "stage5_sensitive_removal_comparison.csv",
    index=False
)

print("Sensitive attribute removal comparison saved!")

Sensitive attribute removal comparison saved!


In [45]:
# Fairlearn mitigation setup

from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
from sklearn.ensemble import RandomForestClassifier

print("Fairlearn mitigation imports successful!")
print("Selected model: Random Forest (Class Weight)")

Fairlearn mitigation imports successful!
Selected model: Random Forest (Class Weight)


In [47]:
training_groups = create_fairness_groups(X_train)

print("Training fairness groups created!")
print(training_groups["Gender"].value_counts())
print("Rows:", len(training_groups))
print("Training rows:", len(X_train_unaware_processed))

Training fairness groups created!
Gender
Female    11612
Male       7588
Name: count, dtype: int64
Rows: 19200
Training rows: 19200


In [48]:
base_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

fair_model = ExponentiatedGradient(
    estimator=base_model,
    constraints=EqualizedOdds(),
    eps=0.01
)

fair_model.fit(
    X_train_unaware_processed,
    y_train,
    sensitive_features=training_groups["Gender"]
)

print("Fairlearn mitigation model trained successfully!")

Fairlearn mitigation model trained successfully!


In [49]:
# Generate Fairlearn validation predictions

fair_predictions = fair_model.predict(X_val_unaware_processed)

print("Validation predictions generated!")
print("Prediction shape:", fair_predictions.shape)
print("Predicted classes:")
print(pd.Series(fair_predictions).value_counts().sort_index())

Validation predictions generated!
Prediction shape: (4800,)
Predicted classes:
0    3813
1     987
Name: count, dtype: int64


In [51]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)

fair_predictions = fair_model.predict(X_val_unaware_processed)

fair_performance = pd.DataFrame([{
    "Strategy": "Fairlearn Equalized Odds",
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_val, fair_predictions),
    "Precision": precision_score(y_val, fair_predictions, zero_division=0),
    "Recall": recall_score(y_val, fair_predictions, zero_division=0),
    "F1": f1_score(y_val, fair_predictions, zero_division=0),
    "ROC_AUC": roc_auc_score(y_val, fair_predictions)
}])

display(fair_performance.round(4))

print("Confusion Matrix:")
print(confusion_matrix(y_val, fair_predictions))

,Strategy,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Fairlearn Equalized Odds,Random Forest,0.7919,0.5319,0.4944,0.5124,0.6854


Confusion Matrix:
[[3276  462]
 [ 537  525]]


In [53]:
# Find fairness-related functions already defined in the notebook

print([
    name for name, obj in globals().items()
    if callable(obj) and "fair" in name.lower()
])

['create_fairness_groups']


In [54]:
def calculate_fairness_metrics(y_true, y_pred, groups):
    records = []

    audit_data = groups.copy()
    audit_data["Actual"] = np.asarray(y_true)
    audit_data["Predicted"] = np.asarray(y_pred)

    for dimension in groups.columns:
        for group_name, subset in audit_data.groupby(
            dimension, observed=True
        ):
            tp = ((subset["Actual"] == 1) & (subset["Predicted"] == 1)).sum()
            fp = ((subset["Actual"] == 0) & (subset["Predicted"] == 1)).sum()
            tn = ((subset["Actual"] == 0) & (subset["Predicted"] == 0)).sum()
            fn = ((subset["Actual"] == 1) & (subset["Predicted"] == 0)).sum()

            records.append({
                "Dimension": dimension,
                "Group": group_name,
                "Count": len(subset),
                "Selection_Rate": (tp + fp) / len(subset),
                "TPR": tp / (tp + fn) if (tp + fn) else np.nan,
                "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
                "FNR": fn / (tp + fn) if (tp + fn) else np.nan
            })

    group_df = pd.DataFrame(records)

    gap_records = []

    for dimension, subset in group_df.groupby("Dimension"):
        gap_records.append({
            "Dimension": dimension,
            "DP_Gap": subset["Selection_Rate"].max() - subset["Selection_Rate"].min(),
            "EO_Gap": (
                subset["TPR"].max() - subset["TPR"].min()
                if subset["TPR"].notna().all() else np.nan
            ),
            "FPR_Gap": (
                subset["FPR"].max() - subset["FPR"].min()
                if subset["FPR"].notna().all() else np.nan
            )
        })

    return group_df, pd.DataFrame(gap_records)


print("Fairness calculation function created!")

Fairness calculation function created!


In [55]:
fair_group_df, fair_gaps_df = calculate_fairness_metrics(
    y_val,
    fair_predictions,
    validation_groups
)

print("Group fairness records:", fair_group_df.shape)
print("Fairness gap records:", fair_gaps_df.shape)

display(fair_gaps_df.round(4))

Group fairness records: (24, 7)
Fairness gap records: (5, 4)


,Dimension,DP_Gap,EO_Gap,FPR_Gap
0,Age,0.0569,0.1338,0.0295
1,Education,0.1725,0.2765,0.0868
2,Gender,0.0372,0.0086,0.0242
3,Gender × Age,0.1298,0.3267,0.1378
4,Marital Status,0.0186,0.0458,0.0035


In [56]:
fair_calibration_records = []

audit_data = validation_groups.copy()
audit_data["Actual"] = np.asarray(y_val)
audit_data["Probability"] = fair_model.predict(X_val_unaware_processed)

for dimension in validation_groups.columns:
    for group_name, subset in audit_data.groupby(dimension, observed=True):
        fair_calibration_records.append({
            "Dimension": dimension,
            "Group": group_name,
            "Count": len(subset),
            "Mean_Prediction": subset["Probability"].mean(),
            "Actual_Default_Rate": subset["Actual"].mean(),
            "Calibration_Error": abs(
                subset["Probability"].mean() - subset["Actual"].mean()
            )
        })

fair_calibration_df = pd.DataFrame(fair_calibration_records)

fair_calibration_gaps = (
    fair_calibration_df.groupby("Dimension")["Calibration_Error"]
    .agg(lambda x: x.max() - x.min())
    .reset_index(name="Calibration_Gap")
)

display(fair_calibration_gaps.round(4))

,Dimension,Calibration_Gap
0,Age,0.0521
1,Education,0.0248
2,Gender,0.0186
3,Gender × Age,0.1657
4,Marital Status,0.0191


In [57]:
print("Available Fairlearn attributes:")
print([name for name in dir(fair_model) if not name.startswith("_")])

print("\nNumber of predictors:", len(fair_model.predictors_))
print("Mixture weights:")
print(fair_model.weights_)

Available Fairlearn attributes:
['best_gap_', 'best_iter_', 'constraints', 'constraints_', 'eps', 'estimator', 'eta0', 'fit', 'get_metadata_routing', 'get_params', 'lambda_vecs_', 'lambda_vecs_EG_', 'lambda_vecs_LP_', 'last_iter_', 'max_iter', 'n_oracle_calls_', 'n_oracle_calls_dummy_returned_', 'nu', 'objective', 'oracle_execution_times_', 'predict', 'predictors_', 'run_linprog_step', 'sample_weight_name', 'set_params', 'set_predict_request', 'weights_']

Number of predictors: 6
Mixture weights:
0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
5    1.0
dtype: float64


In [58]:
selected_predictor = fair_model.predictors_[5]

print("Predictor type:", type(selected_predictor))
print("Has predict_proba:", hasattr(selected_predictor, "predict_proba"))

Predictor type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Has predict_proba: True


In [59]:
# Obtain probability estimates from the selected predictor
fair_probabilities = selected_predictor.predict_proba(
    X_val_unaware_processed
)[:, 1]

fair_calibration_records = []

audit_data = validation_groups.copy()
audit_data["Actual"] = np.asarray(y_val)
audit_data["Probability"] = fair_probabilities

for dimension in validation_groups.columns:
    for group_name, subset in audit_data.groupby(
        dimension, observed=True
    ):
        fair_calibration_records.append({
            "Dimension": dimension,
            "Group": group_name,
            "Count": len(subset),
            "Mean_Prediction": subset["Probability"].mean(),
            "Actual_Default_Rate": subset["Actual"].mean(),
            "Calibration_Error": abs(
                subset["Probability"].mean() -
                subset["Actual"].mean()
            )
        })

fair_calibration_df = pd.DataFrame(fair_calibration_records)

fair_calibration_gaps = (
    fair_calibration_df.groupby("Dimension")["Calibration_Error"]
    .agg(lambda x: x.max() - x.min())
    .reset_index(name="Calibration_Gap")
)

display(fair_calibration_gaps.round(4))

,Dimension,Calibration_Gap
0,Age,0.0695
1,Education,0.0647
2,Gender,0.0295
3,Gender × Age,0.1285
4,Marital Status,0.0590


In [60]:
# Create a copy of validation data and flip Gender
X_val_counterfactual = X_val.copy()

X_val_counterfactual["SEX"] = X_val_counterfactual["SEX"].map({
    1: 2,
    2: 1
})

# Remove protected attributes, as done during training
X_val_counterfactual_unaware = X_val_counterfactual.drop(
    columns=["SEX", "AGE", "EDUCATION", "MARRIAGE"]
)

# Apply the same preprocessing
X_val_counterfactual_processed = preprocessor_unaware.transform(
    X_val_counterfactual_unaware
)

# Compare probabilities
original_probabilities = fair_probabilities

counterfactual_probabilities = selected_predictor.predict_proba(
    X_val_counterfactual_processed
)[:, 1]

probability_changes = np.abs(
    original_probabilities - counterfactual_probabilities
)

counterfactual_results = pd.DataFrame([{
    "Mean_Probability_Change": probability_changes.mean(),
    "Max_Probability_Change": probability_changes.max(),
    "Prediction_Flip_Rate": np.mean(
        (original_probabilities >= 0.5) !=
        (counterfactual_probabilities >= 0.5)
    )
}])

display(counterfactual_results.round(6))

,Mean_Probability_Change,Max_Probability_Change,Prediction_Flip_Rate
0,0.0,0.0,0.0


In [61]:
# Extract intersectional fairness gaps

intersectional_fairness = fair_gaps_df[
    fair_gaps_df["Dimension"] == "Gender × Age"
].copy()

print("Fairlearn Intersectional Fairness:")
display(intersectional_fairness.round(4))

Fairlearn Intersectional Fairness:


,Dimension,DP_Gap,EO_Gap,FPR_Gap
3,Gender × Age,0.1298,0.3267,0.1378


In [62]:
# Individual Fairness using nearest neighbours

from sklearn.neighbors import NearestNeighbors

# Use standardized non-protected validation features
X_individual = X_val_unaware_processed

neighbors = NearestNeighbors(n_neighbors=6, metric="euclidean")
neighbors.fit(X_individual)

distances, indices = neighbors.kneighbors(X_individual)

individual_probability_differences = []

for i in range(len(X_individual)):
    # Exclude the individual themself (first neighbour)
    neighbour_indices = indices[i, 1:]

    differences = np.abs(
        fair_probabilities[i] -
        fair_probabilities[neighbour_indices]
    )

    individual_probability_differences.extend(differences)

individual_fairness_results = pd.DataFrame([{
    "Mean_Probability_Difference": np.mean(
        individual_probability_differences
    ),
    "Max_Probability_Difference": np.max(
        individual_probability_differences
    )
}])

display(individual_fairness_results.round(4))

,Mean_Probability_Difference,Max_Probability_Difference
0,0.1126,0.94


In [63]:
# Save Fairlearn mitigation results

fair_performance.to_csv(
    results_path / "stage5_fairlearn_performance.csv",
    index=False
)

fair_group_df.to_csv(
    results_path / "stage5_fairlearn_group_fairness.csv",
    index=False
)

fair_gaps_df.to_csv(
    results_path / "stage5_fairlearn_fairness_gaps.csv",
    index=False
)

fair_calibration_df.to_csv(
    results_path / "stage5_fairlearn_calibration.csv",
    index=False
)

fair_calibration_gaps.to_csv(
    results_path / "stage5_fairlearn_calibration_gaps.csv",
    index=False
)

counterfactual_results.to_csv(
    results_path / "stage5_fairlearn_counterfactual.csv",
    index=False
)

intersectional_fairness.to_csv(
    results_path / "stage5_fairlearn_intersectional.csv",
    index=False
)

individual_fairness_results.to_csv(
    results_path / "stage5_fairlearn_individual.csv",
    index=False
)

print("All Fairlearn mitigation results saved successfully!")


All Fairlearn mitigation results saved successfully!


In [64]:
# Extract the Class Weight Random Forest baseline
baseline_rf = baseline_df[
    (baseline_df["Strategy"] == "Class Weight") &
    (baseline_df["Model"] == "Random Forest")
].copy()

# Extract the same model after sensitive attribute removal
unaware_rf = sensitive_removed_df[
    (sensitive_removed_df["Strategy"] == "Class Weight") &
    (sensitive_removed_df["Model"] == "Random Forest")
].copy()

print("Baseline RF:", baseline_rf.shape)
print("Sensitive-removed RF:", unaware_rf.shape)
print("Fairlearn RF:", fair_performance.shape)

Baseline RF: (1, 9)
Sensitive-removed RF: (1, 9)
Fairlearn RF: (1, 7)


In [65]:
# Prepare performance comparison

performance_comparison = pd.concat([
    baseline_rf.assign(Approach="Baseline"),
    unaware_rf.assign(Approach="Sensitive Attribute Removal"),
    fair_performance.assign(Approach="Fairlearn Equalized Odds")
], ignore_index=True)

performance_columns = [
    "Approach",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

display(
    performance_comparison[
        performance_columns
    ].round(4)
)

,Approach,Accuracy,Precision,Recall,F1,ROC_AUC
0,Baseline,0.8027,0.5620,0.4906,0.5239,NaN
1,Sensitive Attribute Removal,0.7900,0.5270,0.4962,0.5112,NaN
2,Fairlearn Equalized Odds,0.7919,0.5319,0.4944,0.5124,0.6854


In [66]:
# Select Random Forest fairness summaries

baseline_fairness_rf = fairness_summary[
    (fairness_summary["Strategy"] == "Class Weight") &
    (fairness_summary["Model"] == "Random Forest")
].copy()

unaware_fairness_rf = sensitive_removed_fairness_summary[
    (sensitive_removed_fairness_summary["Strategy"] == "Class Weight") &
    (sensitive_removed_fairness_summary["Model"] == "Random Forest")
].copy()

print("Baseline fairness:", baseline_fairness_rf.shape)
print("Sensitive removal fairness:", unaware_fairness_rf.shape)

Baseline fairness: (1, 12)
Sensitive removal fairness: (1, 12)


In [67]:
fairlearn_fairness_summary = pd.DataFrame([{
    "Strategy": "Fairlearn Equalized Odds",
    "Model": "Random Forest",
    "Max_DP_Gap": fair_gaps_df["DP_Gap"].max(),
    "Max_EO_Gap": fair_gaps_df["EO_Gap"].max(),
    "Min_Disparate_Impact": np.nan,
    "Max_Calibration_Error_Gap": fair_calibration_gaps["Calibration_Gap"].max(),
    "Intersectional_DP_Gap": intersectional_fairness["DP_Gap"].iloc[0],
    "Intersectional_EO_Gap": intersectional_fairness["EO_Gap"].iloc[0],
    "Mean_Probability_Change": counterfactual_results["Mean_Probability_Change"].iloc[0],
    "Prediction_Flip_Rate": counterfactual_results["Prediction_Flip_Rate"].iloc[0],
    "Mean_Probability_Difference": individual_fairness_results["Mean_Probability_Difference"].iloc[0],
    "Prediction_Disagreement_Rate": np.nan
}])

print("Fairlearn fairness summary prepared!")
display(fairlearn_fairness_summary.round(4))

Fairlearn fairness summary prepared!


,Strategy,Model,Max_DP_Gap,Max_EO_Gap,Min_Disparate_Impact,Max_Calibration_Error_Gap,Intersectional_DP_Gap,Intersectional_EO_Gap,Mean_Probability_Change,Prediction_Flip_Rate,Mean_Probability_Difference,Prediction_Disagreement_Rate
0,Fairlearn Equalized Odds,Random Forest,0.1725,0.3267,NaN,0.1285,0.1298,0.3267,0.0,0.0,0.1126,NaN


In [68]:
# 1. Disparate Impact
# Favourable outcome = predicted non-default (class 0)

di_values = []

for dimension, subset in fair_group_df.groupby("Dimension"):
    favourable_rates = 1 - subset["Selection_Rate"]

    if favourable_rates.max() > 0:
        di = favourable_rates.min() / favourable_rates.max()
        di_values.append(di)

min_disparate_impact = min(di_values) if di_values else np.nan


# 2. Prediction Disagreement Rate
# Use the same nearest neighbours as Individual Fairness

neighbour_indices = indices[:, 1:]

fair_prediction_disagreements = []

for i in range(len(X_individual)):
    neighbour_predictions = fair_predictions[neighbour_indices[i]]

    disagreements = (
        fair_predictions[i] != neighbour_predictions
    )

    fair_prediction_disagreements.extend(disagreements)

prediction_disagreement_rate = np.mean(
    fair_prediction_disagreements
)


# Update Fairlearn summary
fairlearn_fairness_summary.loc[
    0, "Min_Disparate_Impact"
] = min_disparate_impact

fairlearn_fairness_summary.loc[
    0, "Prediction_Disagreement_Rate"
] = prediction_disagreement_rate

print("Missing fairness metrics calculated!")
display(fairlearn_fairness_summary.round(4))

Missing fairness metrics calculated!


,Strategy,Model,Max_DP_Gap,Max_EO_Gap,Min_Disparate_Impact,Max_Calibration_Error_Gap,Intersectional_DP_Gap,Intersectional_EO_Gap,Mean_Probability_Change,Prediction_Flip_Rate,Mean_Probability_Difference,Prediction_Disagreement_Rate
0,Fairlearn Equalized Odds,Random Forest,0.1725,0.3267,0.8112,0.1285,0.1298,0.3267,0.0,0.0,0.1126,0.0992


In [69]:
# Final fairness comparison

baseline_comparison = baseline_fairness_rf.assign(
    Approach="Baseline"
)

unaware_comparison = unaware_fairness_rf.assign(
    Approach="Sensitive Attribute Removal"
)

fairlearn_comparison = fairlearn_fairness_summary.copy()
fairlearn_comparison["Approach"] = "Fairlearn Equalized Odds"

final_fairness_comparison = pd.concat(
    [
        baseline_comparison,
        unaware_comparison,
        fairlearn_comparison
    ],
    ignore_index=True
)

comparison_columns = [
    "Approach",
    "Max_DP_Gap",
    "Max_EO_Gap",
    "Min_Disparate_Impact",
    "Max_Calibration_Error_Gap",
    "Intersectional_DP_Gap",
    "Intersectional_EO_Gap",
    "Mean_Probability_Change",
    "Prediction_Flip_Rate",
    "Mean_Probability_Difference",
    "Prediction_Disagreement_Rate"
]

display(
    final_fairness_comparison[
        comparison_columns
    ].round(4)
)

,Approach,Max_DP_Gap,Max_EO_Gap,Min_Disparate_Impact,Max_Calibration_Error_Gap,Intersectional_DP_Gap,Intersectional_EO_Gap,Mean_Probability_Change,Prediction_Flip_Rate,Mean_Probability_Difference,Prediction_Disagreement_Rate
0,Baseline,0.2032,0.2566,0.7863,0.1478,0.1325,0.2255,NaN,NaN,NaN,NaN
1,Sensitive Attribute Removal,0.1667,0.3267,0.8000,0.1342,0.1667,0.3267,NaN,NaN,NaN,NaN
2,Fairlearn Equalized Odds,0.1725,0.3267,0.8112,0.1285,0.1298,0.3267,0.0,0.0,0.1126,0.0992
